# Gold Hospitality Skills Mart

**Purpose**: Hospitality-specific skill demand trends and patterns.

**Target Table**: `workspace.gold.gold_hospitality_skills`

**Grain**: One row per date per skill within hospitality sector

**Sector Filter**: Filters to hospitality sector_sk values only

**Key Metrics**:
- Skill mention frequency
- Unique jobs requiring each skill
- Skill demand trends over time
- Role-specific skill patterns

**Data Sources**:
- `workspace.warehouse.bridge_job_skill`
- `workspace.warehouse.fact_job_postings` (filtered to hospitality)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hospitality_skills
USING DELTA
COMMENT 'Hospitality sector-specific skill demand trends'
AS

WITH hospitality_sector AS (
  SELECT sector_sk
  FROM workspace.warehouse.dim_sector
  WHERE sector_name IN ('Hospitality', 'Hotels & Resorts', 'Restaurants', 'Food & Beverage')
     OR sector_family = 'Hospitality'
),

hospitality_jobs AS (
  SELECT
    f.posting_date_sk,
    f.job_sk,
    f.sector_sk,
    f.role_sk,
    f.location_sk,
    f.active_flag
  FROM workspace.warehouse.fact_job_postings f
  INNER JOIN hospitality_sector hs ON f.sector_sk = hs.sector_sk
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.active_flag = TRUE
),

skill_facts AS (
  SELECT
    hj.posting_date_sk AS hospitality_date_sk,
    hj.sector_sk,
    hj.role_sk,
    hj.location_sk,
    bjs.skill_sk,
    bjs.job_sk,
    bjs.confidence
  FROM hospitality_jobs hj
  JOIN workspace.warehouse.bridge_job_skill bjs 
    ON hj.job_sk = bjs.job_sk
  WHERE bjs.is_current = TRUE
),

base_metrics AS (
  SELECT
    sf.hospitality_date_sk,
    sf.sector_sk,
    sf.role_sk,
    sf.skill_sk,
    
    -- MEASURES
    COUNT(*) AS mentions_count,
    COUNT(DISTINCT sf.job_sk) AS unique_jobs_count,
    ROUND(AVG(sf.confidence), 4) AS avg_confidence,
    COUNT(DISTINCT sf.location_sk) AS unique_locations
    
  FROM skill_facts sf
  GROUP BY sf.hospitality_date_sk, sf.sector_sk, sf.role_sk, sf.skill_sk
),

-- Aggregation: Overall hospitality skill demand (all roles)
hospitality_skill_total AS (
  SELECT
    hospitality_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    skill_sk,
    'overall' AS aggregation_level,
    SUM(mentions_count) AS mentions_count,
    SUM(unique_jobs_count) AS unique_jobs_count,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence,
    MAX(unique_locations) AS unique_locations
  FROM base_metrics
  GROUP BY hospitality_date_sk, sector_sk, skill_sk
),

-- Aggregation: By role within hospitality
hospitality_skill_by_role AS (
  SELECT
    hospitality_date_sk,
    sector_sk,
    role_sk,
    skill_sk,
    'by_role' AS aggregation_level,
    SUM(mentions_count) AS mentions_count,
    SUM(unique_jobs_count) AS unique_jobs_count,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence,
    MAX(unique_locations) AS unique_locations
  FROM base_metrics
  GROUP BY hospitality_date_sk, sector_sk, role_sk, skill_sk
),

combined_agg AS (
  SELECT * FROM hospitality_skill_total
  UNION ALL
  SELECT * FROM hospitality_skill_by_role
),

temporal_enriched AS (
  SELECT
    ca.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(ca.mentions_count, 7) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.skill_sk
      ORDER BY ca.hospitality_date_sk
    ) AS prev_week_mentions,
    
    LAG(ca.mentions_count, 30) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.skill_sk
      ORDER BY ca.hospitality_date_sk
    ) AS prev_month_mentions,
    
    -- 7-day rolling average
    AVG(ca.mentions_count) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.skill_sk
      ORDER BY ca.hospitality_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_mentions,
    
    -- 30-day rolling total
    SUM(ca.mentions_count) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.skill_sk
      ORDER BY ca.hospitality_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_total_mentions,
    
    -- Rank skills by mentions within each day
    DENSE_RANK() OVER (
      PARTITION BY ca.hospitality_date_sk, ca.aggregation_level, ca.sector_sk, ca.role_sk
      ORDER BY ca.mentions_count DESC
    ) AS daily_skill_rank
    
  FROM combined_agg ca
)

SELECT
  -- DIMENSIONS
  te.hospitality_date_sk,
  te.sector_sk,
  te.role_sk,
  te.skill_sk,
  te.aggregation_level,
  
  -- MEASURES
  te.mentions_count,
  te.unique_jobs_count,
  te.avg_confidence,
  te.unique_locations,
  te.daily_skill_rank,
  
  -- TEMPORAL METRICS
  CAST(te.rolling_7day_avg_mentions AS DECIMAL(10,2)) AS rolling_7day_avg_mentions,
  te.rolling_30day_total_mentions,
  
  -- DERIVED: Change vs previous periods
  CAST((te.mentions_count - COALESCE(te.prev_week_mentions, 0)) AS BIGINT) AS delta_vs_prev_week,
  CAST((te.mentions_count - COALESCE(te.prev_month_mentions, 0)) AS BIGINT) AS delta_vs_prev_month,
  
  -- DERIVED: Percent change
  CASE 
    WHEN te.prev_week_mentions > 0 
    THEN CAST((te.mentions_count - te.prev_week_mentions) AS DECIMAL(10,4)) / te.prev_week_mentions
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_mentions > 0 
    THEN CAST((te.mentions_count - te.prev_month_mentions) AS DECIMAL(10,4)) / te.prev_month_mentions
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Skill demand intensity (mentions per unique job)
  CASE 
    WHEN te.unique_jobs_count > 0 
    THEN CAST(te.mentions_count AS DECIMAL(10,2)) / te.unique_jobs_count
    ELSE NULL 
  END AS skill_intensity,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
WHERE te.daily_skill_rank <= 100  -- Keep top 100 skills per day for dashboard performance
ORDER BY te.aggregation_level, te.hospitality_date_sk DESC, te.mentions_count DESC;

In [0]:
%sql
-- Validation: Top hospitality skills
SELECT
  s.skill_name,
  s.skill_category,
  SUM(ghs.mentions_count) AS total_mentions,
  SUM(ghs.unique_jobs_count) AS total_unique_jobs,
  ROUND(AVG(ghs.avg_confidence), 4) AS overall_avg_confidence,
  ROUND(AVG(ghs.skill_intensity), 2) AS avg_skill_intensity,
  COUNT(DISTINCT ghs.hospitality_date_sk) AS days_with_demand
FROM workspace.gold.gold_hospitality_skills ghs
JOIN workspace.warehouse.dim_skill s ON ghs.skill_sk = s.skill_sk
WHERE ghs.aggregation_level = 'overall'
GROUP BY s.skill_name, s.skill_category
ORDER BY total_mentions DESC
LIMIT 25;